# 02 Parse Bronze Tables

Read downloaded ZIPs from the raw manifest, parse MMSDM CSV content, append Bronze Delta tables, write file audit rows, and quarantine malformed files.

## Configure Bronze Parse Run

This cell detects local versus Fabric runtime and defines parse parameters.

In [1]:
# Cell purpose: Configure Bronze Parse Run.
from datetime import datetime, timezone
from pathlib import Path
import csv
import importlib
import os
import sys
import uuid

try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

run_id = str(uuid.uuid4())
max_files_per_run = 500
fabric_quarantine_path = "Files/nemweb/quarantine"
local_output_folder = "data"
local_quarantine_path = "files/nemweb/quarantine"

print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")

run_id=c809d79a-cba2-4b18-8c9f-4d5d6a29aa72
runtime=local


## Resolve Runtime Paths

This cell resolves local or Fabric package paths before importing project modules.

In [ ]:
# Cell purpose: Resolve Runtime Paths.
def find_repo_root(start: Path) -> Path:
    """Find the local repo root from a notebook working directory."""

    for candidate in (start, *start.parents):
        if (candidate / "src" / "nem_fabric").exists():
            return candidate
    return start


repo_root = None
local_output_root = None
package_paths = []

if is_local_run:
    repo_root = find_repo_root(Path.cwd())
    local_output_root = repo_root / local_output_folder
    package_paths.append(repo_root / "src")
else:
    package_paths.append(
        Path(os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs"))
    )

for package_path in package_paths:
    if package_path.exists() and str(package_path) not in sys.path:
        sys.path.insert(0, str(package_path))

print(
    "Python search paths added:", [str(path) for path in package_paths if path.exists()]
)
if is_local_run:
    print(f"Local output root: {local_output_root}")

Python search paths added: ['c:\\Users\\brcol\\My Drive\\Documents\\!!!Resume\\Sample Work\\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\\src']
Local output root: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data


## Import Parser and Define Helpers

This cell imports project modules after runtime paths are resolved, then defines local and Fabric helper functions.

In [ ]:
# Cell purpose: Import Parser and Define Helpers.
import pandas as pd

from nem_fabric.common_mmsdm_parser import parse_zip_bytes

local_ingestion = (
    importlib.import_module("nem_fabric.local_ingestion") if is_local_run else None
)

if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
else:
    pass


def table_exists(table_name: str) -> bool:
    """Return True when a Lakehouse table exists in the current Spark catalogue."""

    return spark.catalog.tableExists(table_name)


def read_binary(relative_path: str) -> bytes:
    """Read ZIP bytes from the selected runtime storage root."""

    root = local_output_root if is_local_run else Path("/lakehouse/default")
    if root is None:
        raise RuntimeError("Local output root was not resolved.")
    with (root / relative_path).open("rb") as file:
        return file.read()


def append_local_rows(path: Path, rows: list[dict]) -> None:
    """Append local CSV rows using the project helper."""

    if local_ingestion is None:
        raise RuntimeError("local_ingestion was not imported for local run.")
    local_ingestion.append_csv_rows(path, rows)


if not is_local_run and not table_exists("nem_raw_zip_manifest"):
    raise RuntimeError("nem_raw_zip_manifest does not exist. Run notebook 01 first.")

## Select Unparsed ZIP Files

This cell reads the raw ZIP manifest and removes files already present in the file audit table, making Bronze parsing idempotent.

In [ ]:
# Cell purpose: Select Unparsed ZIP Files.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    manifest_path = local_output_root / "tables" / "nem_raw_zip_manifest.csv"
    audit_path = local_output_root / "tables" / "nem_raw_file_audit.csv"
    if not manifest_path.exists():
        raise RuntimeError("Local manifest does not exist. Run notebook 01 first.")
    with manifest_path.open("r", encoding="utf-8", newline="") as file:
        manifest_rows = [
            row for row in csv.DictReader(file) if row.get("status") == "downloaded"
        ]
    parsed_urls = set()
    if audit_path.exists():
        with audit_path.open("r", encoding="utf-8", newline="") as file:
            parsed_urls = {
                row["source_url"]
                for row in csv.DictReader(file)
                if row.get("source_url")
            }
    files_to_parse = [
        row for row in manifest_rows if row["source_url"] not in parsed_urls
    ]
    files_to_parse = sorted(
        files_to_parse, key=lambda row: row.get("file_datetime", "")
    )[:max_files_per_run]
else:
    manifest = spark.table("nem_raw_zip_manifest").filter("status = 'downloaded'")
    if table_exists("nem_raw_file_audit"):
        parsed = spark.table("nem_raw_file_audit").select("source_url").distinct()
        manifest = manifest.join(parsed, on="source_url", how="left_anti")
    files_to_parse = (
        manifest.orderBy("file_datetime").limit(max_files_per_run).collect()
    )

print(f"Files selected for Bronze parsing: {len(files_to_parse)}")

Files selected for Bronze parsing: 196


## Parse ZIP Files and Build Audit Records

This cell parses each selected ZIP, preserves row-level metadata, collects Bronze DataFrames, and records parsing outcomes for auditability.

In [ ]:
# Cell purpose: Parse ZIP Files and Build Audit Records.
bronze_frames = []
audit_rows = []

for item in files_to_parse:
    parsed_at = datetime.now(timezone.utc).isoformat()
    status = "parsed"
    error_message = ""
    table_count = 0
    row_count = 0
    source_url = item["source_url"] if is_local_run else item.source_url
    source_name = item["source_name"] if is_local_run else item.source_name
    source_zip_name = item["source_zip_name"] if is_local_run else item.source_zip_name
    lakehouse_path = item["lakehouse_path"] if is_local_run else item.lakehouse_path
    try:
        zip_bytes = read_binary(lakehouse_path)
        tables = parse_zip_bytes(zip_bytes, source_url)
        table_count = len(tables)
        for table in tables:
            pdf = table.dataframe.copy()
            pdf["run_id"] = run_id
            pdf["source_name"] = source_name
            pdf["bronze_loaded_datetime"] = parsed_at
            row_count += len(pdf)
            if not pdf.empty:
                bronze_frames.append(pdf)
    except Exception as exc:
        status = "failed"
        error_message = str(exc)[:4000]
        try:
            root = local_output_root if is_local_run else Path("/lakehouse/default")
            quarantine_path = (
                local_quarantine_path if is_local_run else fabric_quarantine_path
            )
            if root is None:
                raise RuntimeError("Local output root was not resolved.")
            source_file = root / lakehouse_path
            target_file = root / quarantine_path / source_zip_name
            target_file.parent.mkdir(parents=True, exist_ok=True)
            target_file.write_bytes(source_file.read_bytes())
        except Exception as quarantine_exc:
            error_message = f"{error_message}; quarantine_failed={quarantine_exc}"[
                :4000
            ]

    audit_rows.append(
        {
            "run_id": run_id,
            "source_name": source_name,
            "source_url": source_url,
            "source_zip_name": source_zip_name,
            "lakehouse_path": lakehouse_path,
            "parsed_datetime": parsed_at,
            "status": status,
            "table_count": table_count,
            "row_count_bronze": row_count,
            "error_message": error_message,
        }
    )

## Write Bronze and Audit Tables

This cell appends parsed MMSDM rows to Bronze Delta tables, creates source-specific Bronze subsets, and writes file audit results.

In [6]:
# Cell purpose: Write Bronze and Audit Tables.
if bronze_frames:
    bronze_pdf = pd.concat(bronze_frames, ignore_index=True, sort=False).fillna("")

    if is_local_run:
        if local_output_root is None:
            raise RuntimeError("local_output_root was not resolved for local run.")
        append_local_rows(
            local_output_root / "tables" / "nem_bronze_mmsdm_rows.csv",
            bronze_pdf.astype(str).to_dict("records"),
        )
    else:
        bronze_sdf = spark.createDataFrame(bronze_pdf.astype(str))
        bronze_sdf.write.format("delta").mode("append").saveAsTable(
            "nem_bronze_mmsdm_rows"
        )

        for table_name, filter_expr in {
            "nem_bronze_dispatchis": "package_name = 'DISPATCH'",
            "nem_bronze_public_prices": "upper(table_name) like '%PRICE%'",
            "nem_bronze_interconnector": "upper(table_name) like '%INTERCONNECT%'",
            "nem_bronze_generation": "upper(table_name) like '%GEN%' or upper(table_name) like '%SCADA%'",
        }.items():
            subset = bronze_sdf.filter(filter_expr)
            if subset.limit(1).count() > 0:
                subset.write.format("delta").mode("append").saveAsTable(table_name)
else:
    print("No Bronze rows produced.")

if audit_rows:
    if is_local_run:
        if local_output_root is None:
            raise RuntimeError("local_output_root was not resolved for local run.")
        append_local_rows(
            local_output_root / "tables" / "nem_raw_file_audit.csv", audit_rows
        )
        print(f"Audit CSV: {local_output_root / 'tables' / 'nem_raw_file_audit.csv'}")
        print(f"Audit rows written: {len(audit_rows)}")
    else:
        spark.createDataFrame(audit_rows).write.format("delta").mode(
            "append"
        ).saveAsTable("nem_raw_file_audit")
        display(spark.createDataFrame(audit_rows))
else:
    print("No files required parsing.")

Audit CSV: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data\tables\nem_raw_file_audit.csv
Audit rows written: 196
